<a href="https://colab.research.google.com/github/aleenaharoon245-dev/Expense_Tracker/blob/main/expense_tracker1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ===================== EXPENSE TRACKER PRO (Simplified & Fixed) =====================
# Is poore code ko Google Colab ke ek single cell mein paste karke Run karein.

import os, html as _h, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output, HTML

# ------------------------------- SETTINGS & FILES ------------------------------------
DATA_FILE   = "/content/expenses.csv"
CLICK_FILE  = "/content/click_log.csv"
BUDGET_FILE = "/content/budget.csv"

COLS       = ["ID", "Date", "Title", "Category", "Payment", "Amount", "Note"]
CATEGORIES = ["Food", "Transport", "Shopping", "Bills", "Health", "Education", "Entertainment", "Rent", "Other"]
PAYMENTS   = ["Cash", "Card", "Bank Transfer", "JazzCash", "Easypaisa"]
CUR        = "Rs"
PKT        = dt.timezone(dt.timedelta(hours=5))
now        = lambda: dt.datetime.now(PKT)

# ------------------------------- DATA LAYER ------------------------------------------
def load_expenses():
    if os.path.exists(DATA_FILE):
        d = pd.read_csv(DATA_FILE)
        if not d.empty:
            d["ID"] = d["ID"].astype(int)
            d["Amount"] = d["Amount"].astype(float)
            d["Note"] = d["Note"].fillna("")
            return d[COLS]
    return pd.DataFrame(columns=COLS).astype({"ID": int, "Amount": float})

def load_clicks():
    if os.path.exists(CLICK_FILE):
        return pd.read_csv(CLICK_FILE, dtype=str, keep_default_na=False)
    return pd.DataFrame(columns=["Time", "Button", "RecordID"])

def load_budget():
    if os.path.exists(BUDGET_FILE):
        try:
            d = pd.read_csv(BUDGET_FILE)
            return float(d.iloc[0]["Monthly"]), float(d.iloc[0]["Weekly"]), float(d.iloc[0]["Daily"])
        except Exception:
            pass
    return 50000.0, 12000.0, 2000.0

expenses  = load_expenses()
click_log = load_clicks()
budget_m, budget_w, budget_d = load_budget()

def save_expenses(): expenses.to_csv(DATA_FILE, index=False)
def save_clicks():   click_log.to_csv(CLICK_FILE, index=False)
def save_budget():
    pd.DataFrame([{"Monthly": budget_m, "Weekly": budget_w, "Daily": budget_d}]).to_csv(BUDGET_FILE, index=False)

def next_id(): return int(expenses["ID"].max()) + 1 if len(expenses) else 1

def track(button, rid=""):
    click_log.loc[len(click_log)] = [now().strftime("%Y-%m-%d %H:%M:%S"), button, str(rid) if rid else ""]
    save_clicks()

def cnt(name): return int((click_log["Button"] == name).sum())

# ------------------------------- WIDGETS SETUP ---------------------------------------
BTN_STYLE = {
    "Add":            ("Add Expense",     "plus",     "success"),
    "Update":         ("Update Selected", "save",     "info"),
    "Delete":         ("Delete Selected", "trash",    "danger"),
    "Clear Form":     ("Clear Form",      "eraser",   "warning"),
    "Set Budget":     ("Save Budget",     "check",    "success"),
    "Export CSV":     ("Export CSV",      "download", ""),
    "Refresh":        ("Refresh",         "refresh",  "primary"),
    "Reset Expenses": ("Reset Expenses",  "bomb",     "danger"),
    "Reset Clicks":   ("Reset Clicks",    "undo",     "warning"),
}
ALL_ACTIONS = list(BTN_STYLE) + ["Select ID", "Filter"]

btn = {k: w.Button(description=v[0], icon=v[1], button_style=v[2],
                   layout=w.Layout(width="180px", height="38px"))
       for k, v in BTN_STYLE.items()}

S = {"description_width": "100px"}
L = w.Layout(width="320px")

# Simple Expense Form Widgets
sel_w   = w.Dropdown(description="Record ID", options=[("— New entry —", 0)], value=0, style=S, layout=L)
date_w  = w.DatePicker(description="Date", value=now().date(), style=S, layout=L)
title_w = w.Text(description="Title", placeholder="e.g. Lunch / Rent", style=S, layout=L)
cat_w   = w.Dropdown(description="Category", options=CATEGORIES, style=S, layout=L)
amt_w   = w.FloatText(description=f"Amount ({CUR})", value=0.0, style=S, layout=L)
pay_w   = w.Dropdown(description="Payment", options=PAYMENTS, style=S, layout=L)
note_w  = w.Text(description="Note", placeholder="optional", style=S, layout=L)

# Budget Inputs
bg_m_w = w.FloatText(description=f"Monthly ({CUR})", value=budget_m, style=S, layout=L)
bg_w_w = w.FloatText(description=f"Weekly ({CUR})", value=budget_w, style=S, layout=L)
bg_d_w = w.FloatText(description=f"Daily ({CUR})", value=budget_d, style=S, layout=L)

filt_cat = w.Dropdown(description="Filter", options=["All"] + CATEGORIES, value="All",
                      style={"description_width": "50px"}, layout=w.Layout(width="200px"))
search_w = w.Text(description="Search", placeholder="search...", continuous_update=False,
                  style={"description_width": "60px"}, layout=w.Layout(width="220px"))
confirm_w = w.Checkbox(description="Confirm reset", value=False, indent=False, layout=w.Layout(width="130px"))

msg = w.HTML()
home_out, expense_budget_summary, table_out, charts_out, clicks_out = (w.Output(), w.Output(), w.Output(), w.Output(), w.Output())

def say(text, kind="ok"):
    bg = {"ok": "#10b981", "warn": "#f59e0b", "err": "#ef4444", "info": "#3b82f6"}[kind]
    msg.value = (f"<div style='padding:8px 14px;border-radius:8px;background:{bg};color:#fff;"
                 f"font-weight:600;display:inline-block;margin-top:5px;'>{_h.escape(text)}</div>")

# ------------------------------- RENDERING -------------------------------------------
CARD_STYLE = ("background: rgba(255, 255, 255, 0.85); backdrop-filter: blur(8px); "
              "border: 1px solid rgba(255, 255, 255, 0.8); border-radius: 12px; "
              "padding: 12px 16px; box-shadow: 0 4px 12px rgba(0,0,0,0.04); flex:1; min-width:160px;")

def render_budget_status():
    total_spent = expenses["Amount"].sum()
    rem_budget = budget_m - total_spent
    pct = min(100, max(0, (total_spent / budget_m * 100))) if budget_m > 0 else 0

    # Check Out of Budget Condition
    warning_banner = ""
    if total_spent > budget_m:
        warning_banner = f"""
        <div style='background:#f87171; color:#ffffff; padding:10px 16px; border-radius:10px;
                    font-weight:700; font-size:15px; margin-bottom:12px; box-shadow:0 3px 10px rgba(239, 68, 68, 0.3);'>
            ⚠️ Warning: We are out of budget! (Overspent by {CUR} {abs(rem_budget):,.0f})
        </div>
        """

    rem_color = "#10b981" if rem_budget >= 0 else "#ef4444"

    h = f"""
    {warning_banner}
    <div style='display:flex; gap:12px; flex-wrap:wrap; margin-bottom:10px;'>
        <div style='{CARD_STYLE}'>
            <div style='font-size:12px; color:#6b7280; font-weight:600;'>Monthly Budget</div>
            <div style='font-size:20px; font-weight:800; color:#1e293b;'>{CUR} {budget_m:,.0f}</div>
        </div>
        <div style='{CARD_STYLE}'>
            <div style='font-size:12px; color:#6b7280; font-weight:600;'>Used Payment (Spent)</div>
            <div style='font-size:20px; font-weight:800; color:#ef4444;'>{CUR} {total_spent:,.0f}</div>
        </div>
        <div style='{CARD_STYLE}'>
            <div style='font-size:12px; color:#6b7280; font-weight:600;'>Remaining Budget</div>
            <div style='font-size:20px; font-weight:800; color:{rem_color};'>{CUR} {rem_budget:,.0f}</div>
        </div>
    </div>
    <div style='background:rgba(255,255,255,0.85); padding:10px 14px; border-radius:10px; margin-bottom:10px;'>
        <div style='display:flex; justify-content:space-between; margin-bottom:4px; font-weight:600; font-size:13px;'>
            <span>Budget Progress</span>
            <span>{pct:.1f}% Used</span>
        </div>
        <div style='background:#e2e8f0; border-radius:8px; height:10px; overflow:hidden;'>
            <div style='width:{pct}%; background:{"#ef4444" if pct >= 100 else "linear-gradient(90deg, #93c5fd, #f472b6)"}; height:100%;'></div>
        </div>
    </div>
    """
    return h

def render_home():
    total_spent = expenses["Amount"].sum()
    if not expenses.empty:
        last_item = expenses.iloc[-1]
        last_title = _h.escape(str(last_item["Title"]))
        last_amt = f"{CUR} {last_item['Amount']:,.0f}"
        last_cat = last_item["Category"]
    else:
        last_title, last_amt, last_cat = "None", f"{CUR} 0", "-"

    budget_html = render_budget_status()
    last_item_html = f"""
    <div style='{CARD_STYLE} margin-top:10px;'>
        <div style='font-size:12px; color:#6b7280; font-weight:600;'>Last Item Purchased</div>
        <div style='font-size:18px; font-weight:700; color:#3b82f6;'>{last_title}</div>
        <div style='font-size:13px; color:#4b5563;'>Price: <b>{last_amt}</b> | Category: {last_cat}</div>
    </div>
    """

    with home_out:
        clear_output(wait=True)
        display(HTML(budget_html + last_item_html))

def render_expense_budget_summary():
    with expense_budget_summary:
        clear_output(wait=True)
        display(HTML(render_budget_status()))

def render_table():
    df = expenses.copy()
    if filt_cat.value != "All":
        df = df[df["Category"] == filt_cat.value]
    q = search_w.value.strip().lower()
    if q:
        df = df[df["Title"].str.lower().str.contains(q) | df["Note"].str.lower().str.contains(q)]
    df = df.sort_values("ID", ascending=False)

    idc = click_log[click_log["RecordID"] != ""]["RecordID"].value_counts()
    th = "padding:8px 10px; text-align:left; background:rgba(219, 234, 254, 0.8); color:#1e293b; font-size:13px; font-weight:700;"
    td = "padding:7px 10px; border-bottom:1px solid rgba(226, 232, 240, 0.8); font-size:13px; color:#334155;"
    rows = ""
    for r in df.itertuples():
        rows += (f"<tr><td style='{td}'><span style='background:#ec4899; color:#fff; padding:2px 8px; "
                 f"border-radius:10px; font-weight:700;'>#{r.ID}</span></td>"
                 f"<td style='{td}'>{r.Date}</td><td style='{td}; font-weight:600;'>{_h.escape(r.Title)}</td>"
                 f"<td style='{td}'><span style='background:#dbeafe; padding:2px 8px; border-radius:6px;'>{r.Category}</span></td>"
                 f"<td style='{td}'>{r.Payment}</td><td style='{td}; text-align:right; font-weight:700;'>{CUR} {r.Amount:,.0f}</td>"
                 f"<td style='{td}; text-align:center;'><span style='background:#fef3c7; padding:2px 8px; border-radius:8px; font-weight:700;'>"
                 f"{int(idc.get(str(r.ID), 0))}</span></td>"
                 f"<td style='{td}; color:#64748b;'>{_h.escape(str(r.Note))}</td></tr>")
    if not rows:
        rows = f"<tr><td colspan=8 style='{td}; text-align:center; color:#64748b;'>No expenses found</td></tr>"
    head = "".join(f"<th style='{th}'>{c}</th>" for c in ["ID", "Date", "Title", "Category", "Payment", "Amount", "Clicks", "Note"])
    foot = f"<div style='text-align:right; padding:8px; font-weight:700; color:#1e293b;'>Total: {CUR} {df['Amount'].sum():,.0f}</div>"

    with table_out:
        clear_output(wait=True)
        display(HTML(f"<div style='background:rgba(255,255,255,0.85); border-radius:10px; overflow:auto; max-height:300px; "
                     f"box-shadow:0 2px 8px rgba(0,0,0,0.03);'><table style='width:100%; border-collapse:collapse;'>"
                     f"<tr>{head}</tr>{rows}</table>{foot}</div>"))

def render_charts():
    with charts_out:
        clear_output(wait=True)
        fig, axs = plt.subplots(1, 3, figsize=(16, 4))
        fig.patch.set_facecolor('#f8fafc')

        if expenses.empty:
            for a in axs: a.axis("off")
            axs[1].text(.5, .5, "No expense data yet", ha="center", va="center", fontsize=14)
        else:
            c = expenses.groupby("Category")["Amount"].sum().sort_values(ascending=False)
            axs[0].pie(c, labels=c.index, autopct="%1.0f%%", startangle=90,
                       colors=['#f472b6', '#60a5fa', '#34d399', '#fbbf24', '#a78bfa'],
                       wedgeprops=dict(width=.45, edgecolor="white"))
            axs[0].set_title("Spending by Category", fontsize=11, fontweight='bold')

            d = expenses.groupby("Date")["Amount"].sum().tail(14)
            axs[1].bar(d.index, d.values, color="#60a5fa")
            axs[1].set_title("Daily Expenses", fontsize=11, fontweight='bold')
            axs[1].tick_params(axis="x", rotation=45)

            p = expenses.groupby("Payment")["Amount"].sum().sort_values()
            axs[2].barh(p.index, p.values, color="#f472b6")
            axs[2].set_title("Payment Methods", fontsize=11, fontweight='bold')

            for a in axs[1:]:
                a.spines[["top", "right"]].set_visible(False)
        plt.tight_layout(); plt.show()

def render_clicks():
    with clicks_out:
        clear_output(wait=True)
        total = len(click_log)
        th = "padding:8px 10px; background:rgba(219, 234, 254, 0.8); color:#1e293b; text-align:left; font-size:13px;"
        td = "padding:6px 10px; border-bottom:1px solid rgba(226, 232, 240, 0.8); font-size:13px;"
        rows = ""
        for a in ALL_ACTIONS:
            sub = click_log[click_log["Button"] == a]
            n = len(sub)
            pct = (n / total * 100) if total else 0
            rows += (f"<tr><td style='{td}; font-weight:600;'>{a}</td><td style='{td}; text-align:center;'>"
                     f"<b>{n}</b></td><td style='{td}; width:180px;'><div style='background:#e2e8f0; border-radius:6px;'>"
                     f"<div style='width:{pct:.0f}%; background:#a78bfa; height:10px; border-radius:6px;'></div></div></td>"
                     f"<td style='{td}'>{pct:.1f}%</td></tr>")
        t1 = (f"<div style='background:rgba(255,255,255,0.85); border-radius:10px; overflow:hidden; margin-bottom:12px;'>"
              f"<table style='width:100%; border-collapse:collapse;'><tr>"
              + "".join(f"<th style='{th}'>{x}</th>" for x in ["Button / Action", "Clicks", "Share Bar", "%"])
              + f"</tr>{rows}</table></div>")

        display(HTML(f"<h4 style='color:#1e293b; margin:0 0 8px 0;'>Total Clicks Tracked: {total}</h4>{t1}"))
        draw_click_map()

def draw_click_map():
    fig, axs = plt.subplots(1, 2, figsize=(15, 3.8))
    fig.patch.set_facecolor('#f8fafc')

    if click_log.empty:
        for a in axs: a.axis("off")
        axs[0].text(.5, .5, "No clicks logged yet", ha="center", va="center", fontsize=14)
        plt.show(); return

    d = click_log.copy()
    c = d["Button"].value_counts().reindex(ALL_ACTIONS).fillna(0)
    axs[0].barh(c.index[::-1], c.values[::-1], color="#f472b6")
    axs[0].set_title("Clicks per Action / Button", fontsize=11, fontweight='bold')

    ids = d[d["RecordID"] != ""]["RecordID"].value_counts().head(8)
    if len(ids):
        axs[1].bar(["#" + i for i in ids.index], ids.values, color="#60a5fa")
    else:
        axs[1].axis("off"); axs[1].text(.5, .5, "No Record ID Clicks", ha="center", va="center")
    axs[1].set_title("Top Clicked Record IDs", fontsize=11, fontweight='bold')

    for a in axs:
        a.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

_lock = False
def refresh_selector(value=0):
    global _lock
    _lock = True
    ids = [int(i) for i in expenses["ID"]]
    sel_w.options = [("— New entry —", 0)] + [
        (f"#{r.ID} · {r.Title} · {CUR} {r.Amount:,.0f}", int(r.ID))
        for r in expenses.sort_values("ID", ascending=False).itertuples()]
    sel_w.value = value if value in ids else 0
    _lock = False

def render_all():
    for k, b in btn.items():
        if k in BTN_STYLE:
            b.description = f"{BTN_STYLE[k][0]} ({cnt(k)})"
    render_home()
    render_expense_budget_summary()
    render_table()
    render_charts()
    render_clicks()

# ------------------------------- ACTIONS & CONTROLS ----------------------------------
def clear_form():
    title_w.value = ""; amt_w.value = 0.0; note_w.value = ""
    date_w.value = now().date(); cat_w.value = CATEGORIES[0]; pay_w.value = PAYMENTS[0]

def read_form():
    t = title_w.value.strip()
    if not t:               return None, "Please enter a Title"
    if amt_w.value <= 0:    return None, "Amount must be greater than 0"
    return dict(Date=str(date_w.value or now().date()), Title=t, Category=cat_w.value,
                Payment=pay_w.value, Amount=float(amt_w.value), Note=note_w.value.strip()), None

def do_add(_=None):
    global expenses
    data, err = read_form()
    if err:
        track("Add"); say(err, "err"); return render_all()
    nid = next_id()
    expenses.loc[len(expenses)] = [nid, data["Date"], data["Title"], data["Category"],
                                   data["Payment"], data["Amount"], data["Note"]]
    save_expenses(); track("Add", nid); clear_form(); refresh_selector(0)
    say(f"Expense #{nid} added successfully!", "ok"); render_all()

def do_update(_=None):
    rid = sel_w.value
    if not rid:
        track("Update"); say("Please select a Record ID first", "warn"); return render_all()
    data, err = read_form()
    if err:
        track("Update", rid); say(err, "err"); return render_all()
    i = expenses.index[expenses["ID"] == rid][0]
    for k, v in data.items(): expenses.at[i, k] = v
    save_expenses(); track("Update", rid); refresh_selector(rid)
    say(f"Expense #{rid} updated", "info"); render_all()

def do_delete(_=None):
    global expenses
    rid = sel_w.value
    if not rid:
        track("Delete"); say("Please select a Record ID first", "warn"); return render_all()
    expenses = expenses[expenses["ID"] != rid].reset_index(drop=True)
    save_expenses(); track("Delete", rid); clear_form(); refresh_selector(0)
    say(f"Expense #{rid} deleted", "err"); render_all()

def do_clear(_=None):
    clear_form(); refresh_selector(0); track("Clear Form"); say("Form cleared", "info"); render_all()

def do_set_budget(_=None):
    global budget_m, budget_w, budget_d
    budget_m, budget_w, budget_d = float(bg_m_w.value), float(bg_w_w.value), float(bg_d_w.value)
    save_budget(); track("Set Budget"); say("Budget saved successfully!", "ok"); render_all()

def do_export(_=None):
    track("Export CSV")
    path = "/content/expenses_export.csv"
    expenses.to_csv(path, index=False)
    try:
        from google.colab import files
        files.download(path)
    except Exception:
        pass
    say(f"CSV exported!", "ok"); render_all()

def do_refresh(_=None):
    track("Refresh"); say("Refreshed!", "info"); render_all()

def do_reset_exp(_=None):
    global expenses
    track("Reset Expenses")
    if not confirm_w.value:
        say("Check 'Confirm reset' first", "warn"); return render_all()
    expenses = expenses.iloc[0:0].copy(); save_expenses()
    confirm_w.value = False; refresh_selector(0); say("All expenses deleted", "err"); render_all()

def do_reset_clicks(_=None):
    global click_log
    click_log = click_log.iloc[0:0].copy()
    track("Reset Clicks"); say("Click log reset", "info"); render_all()

def on_select(ch):
    if _lock: return
    rid = ch["new"]
    track("Select ID", rid if rid else "")
    if rid:
        r = expenses[expenses["ID"] == rid].iloc[0]
        title_w.value = r["Title"]; cat_w.value = r["Category"]; pay_w.value = r["Payment"]
        amt_w.value = float(r["Amount"]); note_w.value = str(r["Note"])
        try: date_w.value = dt.date.fromisoformat(r["Date"])
        except Exception: pass
        say(f"Loaded Expense #{rid}", "info")
    else:
        clear_form()
    render_all()

def on_filter(_):
    track("Filter"); render_all()

btn["Add"].on_click(do_add);            btn["Update"].on_click(do_update)
btn["Delete"].on_click(do_delete);      btn["Clear Form"].on_click(do_clear)
btn["Set Budget"].on_click(do_set_budget)
btn["Export CSV"].on_click(do_export);  btn["Refresh"].on_click(do_refresh)
btn["Reset Expenses"].on_click(do_reset_exp); btn["Reset Clicks"].on_click(do_reset_clicks)

sel_w.observe(on_select, names="value")
filt_cat.observe(on_filter, names="value"); search_w.observe(on_filter, names="value")

# ------------------------------- LAYOUT & THEME --------------------------------------
display(HTML("""
<style>
    .main-container {
        background: linear-gradient(135deg, #fce7f3 0%, #e0f2fe 50%, #ffffff 100%) !important;
        padding: 20px;
        border-radius: 16px;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    }
    .widget-tab > .p-TabBar .p-TabBar-tab {
        background: rgba(255, 255, 255, 0.6) !important;
        border-radius: 8px 8px 0 0 !important;
        font-weight: 600 !important;
        border: none !important;
    }
    .widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current {
        background: #ffffff !important;
        color: #ec4899 !important;
        border-bottom: 3px solid #ec4899 !important;
    }
</style>
"""))

header = w.HTML("""
<div style='padding:16px 20px; border-radius:12px; color:#1e293b;
            background: linear-gradient(135deg, #fbcfe8, #bae6fd);
            box-shadow:0 2px 8px rgba(0,0,0,0.04); margin-bottom:12px;'>
    <div style='font-size:24px; font-weight:800; color:#831843;'>🌸 Expense Tracker Pro</div>
    <div style='opacity:.85; font-weight:500; font-size:13px;'>Simple & Clean Expense Management</div>
</div>
""")

# Simple Clean Expense Page Layout
simple_form = w.VBox([
    w.HBox([sel_w, date_w], layout=w.Layout(gap="10px")),
    w.HBox([title_w, amt_w], layout=w.Layout(gap="10px")),
    w.HBox([cat_w, pay_w], layout=w.Layout(gap="10px")),
    w.HBox([note_w], layout=w.Layout(gap="10px"))
], layout=w.Layout(gap="6px"))

row1 = w.HBox([btn["Add"], btn["Update"], btn["Delete"], btn["Clear Form"]], layout=w.Layout(gap="8px", flex_flow="row wrap"))
row2 = w.HBox([btn["Refresh"], btn["Export CSV"], btn["Reset Expenses"], confirm_w, btn["Reset Clicks"]],
              layout=w.Layout(gap="8px", flex_flow="row wrap", align_items="center"))

tab_exp = w.VBox([
    expense_budget_summary,
    w.HTML("<h4 style='color:#831843; margin:10px 0 5px 0;'>➕ Add / Edit Expense</h4>"),
    simple_form,
    row1,
    msg,
    w.HTML("<hr style='border:0; border-top:1px solid rgba(0,0,0,0.08); margin:12px 0;'/>"),
    w.HBox([filt_cat, search_w], layout=w.Layout(gap="10px")),
    table_out,
    row2
], layout=w.Layout(gap="8px", padding="10px"))

budget_form = w.VBox([
    w.HBox([bg_m_w, bg_w_w, bg_d_w], layout=w.Layout(gap="10px")),
    btn["Set Budget"]
], layout=w.Layout(gap="10px"))

tab_budget = w.VBox([
    w.HTML("<h4 style='color:#831843;'>🎯 Set Budget Limits</h4>"),
    budget_form,
    w.HTML("<hr style='border:0; border-top:1px solid rgba(0,0,0,0.08); margin:15px 0;'/>"),
    home_out
], layout=w.Layout(padding="10px"))

tabs = w.Tab(children=[
    tab_exp,
    tab_budget,
    w.VBox([charts_out], layout=w.Layout(padding="10px")),
    w.VBox([clicks_out], layout=w.Layout(padding="10px"))
])

titles = ["📝 Expenses", "🏠 Home & Budget", "📊 Analytics", "🖱️ Click Map"]
for i, t in enumerate(titles):
    tabs.set_title(i, t)

refresh_selector(0)
main_view = w.VBox([header, tabs], layout=w.Layout(padding="8px"))
main_view.add_class("main-container")

display(main_view)
render_all()